# 1.Load dataset 

In [1]:
import pandas as pd
df = pd.read_csv('bbc_news.csv')
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(df.isnull().sum())

Total rows: 2224
Total columns: 2
Category    0
Text        0
dtype: int64


In [2]:
# Percentage of missing values per column
missing_percentage = (df.isnull().sum() / len(df)) * 100
print(missing_percentage)
# Rows with any missing values
missing_rows = df[df.isnull().any(axis=1)]
print(missing_rows)


Category    0.0
Text        0.0
dtype: float64
Empty DataFrame
Columns: [Category, Text]
Index: []


In [3]:
# Summary of the dataset
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2224 entries, 0 to 2223
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  2224 non-null   object
 1   Text      2224 non-null   object
dtypes: object(2)
memory usage: 34.9+ KB
None


## 1.1. Preprocessing data

In [4]:
import re
# df['Text'] = df['Text'].apply(lambda x: re.sub(r'[",]', '', x))  # Remove " and ,
texts = df['Text']
labels = df['Category']

In [5]:
print(texts)

0       Ad sales boost Time Warner profit\n\nQuarterly...
1       Dollar gains on Greenspan speech\n\nThe dollar...
2       Yukos unit buyer faces loan claim\n\nThe owner...
3       High fuel prices hit BA's profits\n\nBritish A...
4       Pernod takeover talk lifts Domecq\n\nShares in...
                              ...                        
2219    BT program to beat dialler scams\n\nBT is intr...
2220    Spam e-mails tempt net shoppers\n\nComputer us...
2221    Be careful how you code\n\nA new European dire...
2222    US cyber security chief resigns\n\nThe man mak...
2223    Losing yourself in online gaming\n\nOnline rol...
Name: Text, Length: 2224, dtype: object


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(texts).toarray()
X
X = pd.DataFrame(X)
X

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
0,0.025346,0.0,0.0,0.026598,0.0,0.0,0.0,0.0,0.0,0.034249,...,0.000000,0.0,0.0,0.000000,0.0,0.0000,0.0,0.0,0.0,0.0
1,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.0,0.0000,0.0,0.0,0.0,0.0
2,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.397503,0.0,0.3521,0.0,0.0,0.0,0.0
3,0.020585,0.0,0.0,0.021602,0.0,0.0,0.0,0.0,0.0,0.027816,...,0.000000,0.0,0.0,0.000000,0.0,0.0000,0.0,0.0,0.0,0.0
4,0.000000,0.0,0.0,0.046420,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.0,0.0000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2219,0.049955,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.0,0.0000,0.0,0.0,0.0,0.0
2220,0.025279,0.0,0.0,0.026528,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.0,0.0000,0.0,0.0,0.0,0.0
2221,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.0,0.0000,0.0,0.0,0.0,0.0
2222,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.032642,...,0.000000,0.0,0.0,0.000000,0.0,0.0000,0.0,0.0,0.0,0.0


In [7]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(labels) 
y

array([0, 0, 0, ..., 4, 4, 4])

## 2. Splitting data

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state=42)

# 2.Training

In [9]:
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Dropout
model = Sequential([
    Dense(128,activation='relu',input_dim=X_train.shape[1]),
    Dropout(0.5),
    Dense(64,activation='relu'),
    Dropout(0.5),
    Dense(len(set(y)),activation='softmax')
])

model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

model.fit(X_train,y_train,epochs=10,batch_size=16,validation_data=(X_test,y_test))

loss,accuracy = model.evaluate(X_test,y_test)
print(f'Accuracy:{accuracy}')

c:\Users\Sreylen\miniconda3\envs\lenenv\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.4144 - loss: 1.4842 - val_accuracy: 0.9281 - val_loss: 0.5329
Epoch 2/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9260 - loss: 0.4163 - val_accuracy: 0.9730 - val_loss: 0.1296
Epoch 3/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9821 - loss: 0.1109 - val_accuracy: 0.9730 - val_loss: 0.0995
Epoch 4/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9970 - loss: 0.0484 - val_accuracy: 0.9730 - val_loss: 0.0980
Epoch 5/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9981 - loss: 0.0309 - val_accuracy: 0.9708 - val_loss: 0.1018
Epoch 6/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9999 - loss: 0.0228 - val_accuracy: 0.9663 - val_loss: 0.1021
Epoch 7/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9988 - loss: 0.0125 - val_accuracy: 0.9708 - val_loss: 0.1072
Epoch 8/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9993 - loss: 0.0085 - val_accu

# 3. Model  Saving

In [10]:
model_and_vectorizer = {
    'model': model,         
    'vectorizer': vectorizer  
}


with open('model_and_vectorizer.pkl', 'wb') as f:
    pickle.dump(model_and_vectorizer, f)

print("Model and Vectorizer have been saved.")

NameError: name 'pickle' is not defined

In [50]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

new_document = ['The stock market is experiencing significant growth today.']

X_new = vectorizer.transform(new_document)

predicted_probs = model.predict(X_new)
predicted_class = np.argmax(predicted_probs)

categories = ['business', 'entertainment', 'politics', 'sport', 'tech'] 
print(f'Predicted category: {categories[predicted_class]}')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 324ms/step
Predicted category: business


In [53]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Predict class probabilities
y_pred_prob = model.predict(X_test)

# Convert probabilities to class labels
y_pred = np.argmax(y_pred_prob, axis=1)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(conf_matrix)

# Optionally, display the classification report for more insights
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=encoder.classes_))


14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
Confusion Matrix:
[[107   3   2   0   1]
 [  1  77   0   0   0]
 [  1   0  66   0   2]
 [  1   0   0 100   0]
 [  0   2   0   0  82]]

Classification Report:
               precision    recall  f1-score   support

     business       0.97      0.95      0.96       113
entertainment       0.94      0.99      0.96        78
     politics       0.97      0.96      0.96        69
        sport       1.00      0.99      1.00       101
         tech       0.96      0.98      0.97        84

     accuracy                           0.97       445
    macro avg       0.97      0.97      0.97       445
 weighted avg       0.97      0.97      0.97       445

